# Depth Anything Example

## Depth Anything — Monocular Depth Estimation

This notebook is an example of monocular depth estimation with **Depth Anything V2**.

```
Image (URL or file) → Depth Anything model → Depth map (grayscale / colormap)
```

Monocular depth estimation predicts a **per-pixel depth map** from a single RGB image — no stereo camera or LiDAR required.  
Closer objects appear brighter (or warmer) in the output depth map, farther objects appear darker (or cooler).

### Available models

| Model | Params | Speed |
|-------|--------|-------|
| `depth-anything/Depth-Anything-V2-Large-hf` | 335 M | slowest / most accurate |
| `depth-anything/Depth-Anything-V2-Base-hf`  |  97 M | — |
| `depth-anything/Depth-Anything-V2-Small-hf` |  24 M | fastest / lightest |

Change `MODEL_NAME` in the writefile cell to switch between them.

📄 [Depth Anything V2 paper](https://arxiv.org/abs/2406.09414)  
🤗 [Hugging Face model hub](https://huggingface.co/depth-anything)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/CV-Samples/depth"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!mkdir -p output
!ls


In [ ]:
# Install dependencies
!pip install transformers accelerate -q

import transformers
print(transformers.__version__)  # check version


In [ ]:
# Download sample images
import os

# Image 1: repair scene (from Pixabay CDN)
REPAIR_URL = "https://cdn.pixabay.com/photo/2019/11/04/20/06/repair-4602070_640.jpg"
repair_dest = f'{PROJECT_PATH}/repair-4602070_640.jpg'
if not os.path.exists(repair_dest):
    os.system(f'wget -q "{REPAIR_URL}" -O "{repair_dest}"')
    print('Downloaded: repair-4602070_640.jpg')
else:
    print('Already exists: repair-4602070_640.jpg')

# Image 2: avenue / trees perspective (from GitHub)
BASE_URL = "https://raw.githubusercontent.com/mastnk/cv-samples/main/depth"
IMAGE_FILES = [
    "bertvthul-avenue-815297_640.jpg",
]
for fname in IMAGE_FILES:
    url  = f'{BASE_URL}/{fname}'
    dest = f'{PROJECT_PATH}/{fname}'
    if not os.path.exists(dest):
        os.system(f'wget -q "{url}" -O "{dest}"')
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

%cd "{PROJECT_PATH}"
!ls


## Using Your Own Images

There are two ways to provide images for depth estimation:

**① Specify a URL**  
Pass a direct image URL to the `--url` flag when running the script:
```
%run depth_anything.py --url https://cdn.pixabay.com/photo/2019/11/04/20/06/repair-4602070_640.jpg
```

**② Upload images to the `PROJECT_PATH/` folder**  
Place your image files in `PROJECT_PATH/`, then run the script with `--file` or `--dir`.

The easiest way to upload is through **Google Drive**:  
Open [drive.google.com](https://drive.google.com), navigate to `My Drive / CV-Samples / depth/`, and drag & drop your files there.  
They will be immediately available in Colab via the mounted drive — no extra upload step needed.

```
%run depth_anything.py --file your_image.jpg   # single file
%run depth_anything.py --dir  .                # all images in folder
```

## Selecting a Model

In the writefile cell below, change `MODEL_NAME` to switch models.  
When multiple `MODEL_NAME = ...` lines are listed, **only the last one takes effect**.

```python
# MODEL_NAME = 'depth-anything/Depth-Anything-V2-Large-hf'  # 335M params — most accurate, slowest
# MODEL_NAME = 'depth-anything/Depth-Anything-V2-Base-hf'   #  97M params
MODEL_NAME   = 'depth-anything/Depth-Anything-V2-Small-hf'  #  24M params — fastest, lightest  ← active
```

Larger models produce more accurate depth maps but take longer to run.  
Start with `Depth-Anything-V2-Small-hf` for quick experiments.

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile depth_anything.py
"""Depth Anything V2 — command-line interface.

Usage:
  %run depth_anything.py --url  <image_url>  --out <output_path> [--disp] [--cmap COLORMAP]
  %run depth_anything.py --file <image_path> --out <output_path> [--disp] [--cmap COLORMAP]
  %run depth_anything.py --dir  <image_dir>  --out <output_dir>  [--disp] [--cmap COLORMAP]
"""

import argparse
import glob
import os

import numpy as np
from PIL import Image
import requests
from io import BytesIO
import matplotlib.cm as cm
from transformers import pipeline

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
MODEL_NAME = 'depth-anything/Depth-Anything-V2-Small-hf'  # fastest / lightest
# MODEL_NAME = 'depth-anything/Depth-Anything-V2-Base-hf'   #  97M params
# MODEL_NAME = 'depth-anything/Depth-Anything-V2-Large-hf'  # most accurate

PROJECT_PATH = '/content/drive/MyDrive/CV-Samples/depth'

# ---- Model loading -----------------------------------------------
pipe = pipeline(
    task='depth-estimation',
    model=MODEL_NAME,
    device=0 if __import__('torch').cuda.is_available() else -1,
)

# ---- Depth map helper --------------------------------------------
def depth_to_image(depth_array: np.ndarray, cmap: str = 'inferno') -> Image.Image:
    """Convert a raw depth array to a colorized PIL image."""
    d = depth_array.astype(np.float32)
    d = (d - d.min()) / (d.max() - d.min() + 1e-8)  # normalize 0-1
    colored = (cm.get_cmap(cmap)(d)[:, :, :3] * 255).astype(np.uint8)
    return Image.fromarray(colored)

# ---- Display helper ----------------------------------------------
def show(original: Image.Image, depth_img: Image.Image, label: str, disp: bool) -> None:
    """When --disp is active, display original and depth map side by side."""
    if disp:
        w, h = original.size
        combined = Image.new('RGB', (w * 2, h))
        combined.paste(original, (0, 0))
        combined.paste(depth_img.resize((w, h)), (w, 0))
        if _has_ipy:
            ipy_display(combined)
        print(label)

# ---- Functions ---------------------------------------------------
def estimate_url(url: str, out: str, cmap: str = 'inferno', disp: bool = False):
    """Download an image from a URL and estimate depth."""
    image     = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    result    = pipe(image)
    depth_img = depth_to_image(np.array(result['depth']), cmap)
    show(image, depth_img, url, disp)
    os.makedirs(os.path.dirname(os.path.abspath(out)), exist_ok=True)
    depth_img.save(out)
    print(f'Saved: {out}')
    return result


def estimate_file(path: str, out: str, cmap: str = 'inferno', disp: bool = False):
    """Estimate depth in a single local image file."""
    image     = Image.open(path).convert('RGB')
    result    = pipe(image)
    depth_img = depth_to_image(np.array(result['depth']), cmap)
    show(image, depth_img, path, disp)
    os.makedirs(os.path.dirname(os.path.abspath(out)), exist_ok=True)
    depth_img.save(out)
    print(f'Saved: {out}')
    return result


def estimate_dir(directory: str, out_dir: str, cmap: str = 'inferno', disp: bool = False):
    """Estimate depth in all images in a directory."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    os.makedirs(out_dir, exist_ok=True)
    for path in filepaths:
        base      = os.path.splitext(os.path.basename(path))[0]
        out_path  = os.path.join(out_dir, f'depth_{base}.png')
        estimate_file(path, out_path, cmap, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='Depth Anything V2')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--url',  type=str, help='Image URL for depth estimation')
group.add_argument('--file', type=str, help='Path to a single image file')
group.add_argument('--dir',  type=str, help='Directory of images')
parser.add_argument('--out',  type=str, required=True,
                    help='Output file path (--url/--file) or output directory (--dir)')
parser.add_argument('--cmap', type=str, default='inferno',
                    help='Matplotlib colormap for depth map (default: inferno)')
parser.add_argument('--disp', action='store_true',
                    help='Display original + depth map side by side')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.url:
    estimate_url(args.url,   out=args.out, cmap=args.cmap, disp=args.disp)
elif args.file:
    estimate_file(args.file, out=args.out, cmap=args.cmap, disp=args.disp)
elif args.dir:
    estimate_dir(args.dir,   out_dir=args.out, cmap=args.cmap, disp=args.disp)


## `depth_anything.py` Usage

After running the `%%writefile` cell above, `depth_anything.py` is saved in `PROJECT_PATH`.
Run it with **`%run`** (not `!python`) so that inline image display works in Colab.

```
%run depth_anything.py --url  <image_url>  --out <output_path>  # depth from remote image
%run depth_anything.py --file <image_path> --out <output_path>  # depth from single local file
%run depth_anything.py --dir  <image_dir>  --out <output_dir>   # depth from all images in folder
```

**Required argument**

| Flag | Description |
|------|-------------|
| `--out <path>` | Output file path (`--url` / `--file`) or output directory (`--dir`) |

**Optional arguments**

| Flag | Default | Description |
|------|---------|-------------|
| `--disp` | off | Display original image + depth map side by side |
| `--cmap <name>` | `inferno` | Matplotlib colormap for depth visualization |

**Colormap options** (change `--cmap` to switch)

| Colormap | Appearance |
|----------|-----------|
| `inferno` | black → orange → white (default) |
| `magma`   | black → purple → white |
| `plasma`  | purple → orange → yellow |
| `viridis` | dark blue → green → yellow |
| `gray`    | black → white (classic) |

**Examples**

```bash
# Estimate depth from URL, save to output/
%run depth_anything.py --url https://cdn.pixabay.com/photo/2019/11/04/20/06/repair-4602070_640.jpg --out output/depth_repair.png --disp

# Estimate depth from a local file with gray colormap
%run depth_anything.py --file bertvthul-avenue-815297_640.jpg --out output/depth_avenue.png --cmap gray --disp

# Estimate depth for all images in folder, save all to output/
%run depth_anything.py --dir . --out output/ --disp
```

## Execution Methods

Use **`%run`** to execute `depth_anything.py` inside the Colab kernel — this enables inline image display with `--disp`.

| Mode | Flag | `--out` |
|------|------|---------|
| From URL | `--url <url>` | output **file** path |
| Single file | `--file <path>` | output **file** path |
| Directory | `--dir <path>` | output **directory** path |

`--out` is **required** in all modes.  
Add `--disp` to display original image and depth map side by side.  
Add `--cmap <name>` to change the colormap (default: `inferno`).

In [ ]:
# Execute: estimate depth from URL (with display)
%cd "{PROJECT_PATH}"
%run depth_anything.py --disp --url https://cdn.pixabay.com/photo/2019/11/04/20/06/repair-4602070_640.jpg --out output/depth_repair.png


In [ ]:
# Execute: estimate depth from a single local image file (with display)
%cd "{PROJECT_PATH}"
%run depth_anything.py --disp --file bertvthul-avenue-815297_640.jpg --out output/depth_avenue.png


In [ ]:
# Execute: estimate depth for all images in a directory (with display)
%cd "{PROJECT_PATH}"
%run depth_anything.py --disp --dir . --out output/


💾 **Don't forget to save this notebook!**

To keep your work, go to **File → Save a copy in Drive** before closing.